### 1. Імпорт необхідних бібліотек
Підключення модулів для маніпуляції великими даними (Pandas, Polars, NumPy) та замірів часу виконання.

In [1]:
import os
import time
import numpy as np
import pandas as pd
import polars as pl

### 2. Завантаження CSV, фільтрація та оптимізація типів пам'яті
- Читання лише потрібних стовпців файлу `flights.csv` з фіксацією часу.
- Видалення скасованих (`CANCELLED == 1`) та перенаправлених (`DIVERTED == 1`) рейсів.
- Створення цільової бінарної змінної `is_delayed` (затримка вильоту > 15 хв) та виділення години вильоту `dep_hour`.
- Приведення числових та строкових типів до менших форматів (`int8`, `int16`, `category`) для зменшення використання RAM.

In [2]:
t_start = time.time()
df = pd.read_csv(
    "../flights.csv",
    usecols=[
        "MONTH",
        "DAY_OF_WEEK",
        "AIRLINE",
        "ORIGIN_AIRPORT",
        "DESTINATION_AIRPORT",
        "SCHEDULED_DEPARTURE",
        "DEPARTURE_DELAY",
        "DISTANCE",
        "CANCELLED",
        "DIVERTED",
    ],
    low_memory=False,
)
t_load_csv = round(time.time() - t_start, 2)

df = df[(df["CANCELLED"] == 0) & (df["DIVERTED"] == 0)].copy()
df.dropna(subset=["DEPARTURE_DELAY"], inplace=True)

df["is_delayed"] = (df["DEPARTURE_DELAY"] > 15).astype(np.int8)
df["dep_hour"] = (df["SCHEDULED_DEPARTURE"] // 100).astype(np.int8)

df.drop(
    columns=["CANCELLED", "DIVERTED", "SCHEDULED_DEPARTURE"], inplace=True
)

df["MONTH"] = df["MONTH"].astype(np.int8)
df["DAY_OF_WEEK"] = df["DAY_OF_WEEK"].astype(np.int8)
df["DISTANCE"] = df["DISTANCE"].astype(np.int16)
df["DEPARTURE_DELAY"] = df["DEPARTURE_DELAY"].astype(np.int16)

df["AIRLINE"] = df["AIRLINE"].astype("category")
df["ORIGIN_AIRPORT"] = df["ORIGIN_AIRPORT"].astype(str).astype("category")
df["DESTINATION_AIRPORT"] = (
    df["DESTINATION_AIRPORT"].astype(str).astype("category")
)

df.reset_index(drop=True, inplace=True)
print(f"Дані підготовлено. Розмірність: {df.shape}")

C:\Users\vadim\AppData\Local\Temp\ipykernel_5704\1008281565.py:21: Pandas4Warning: The inplace keyword in DataFrame.dropna is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.dropna(subset=["DEPARTURE_DELAY"], inplace=True)
C:\Users\vadim\AppData\Local\Temp\ipykernel_5704\1008281565.py:26: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(


Дані підготовлено. Розмірність: (5714008, 9)


C:\Users\vadim\AppData\Local\Temp\ipykernel_5704\1008281565.py:41: Pandas4Warning: The inplace keyword in DataFrame.reset_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.reset_index(drop=True, inplace=True)


### 3. Збереження у Parquet та бенчмарк швидкості
Збереження очищених даних у формат Parquet, замір швидкості повторного зчитування через Pandas і Polars та порівняння розмірів файлів на диску.

In [3]:
t_start = time.time()
df.to_parquet("flights_clean.parquet", index=False)
t_save_parquet = round(time.time() - t_start, 2)

t_start = time.time()
df_test = pd.read_parquet("flights_clean.parquet")
t_load_parquet_pd = round(time.time() - t_start, 2)

t_start = time.time()
df_test_pl = pl.read_parquet("flights_clean.parquet")
t_load_parquet_pl = round(time.time() - t_start, 2)

size_csv = round(os.path.getsize("../flights.csv") / (1024 * 1024), 2)
size_parquet = round(os.path.getsize("flights_clean.parquet") / (1024 * 1024), 2)

benchmarks = pd.DataFrame(
    {
        "Операція / Формат": [
            "Читання flights.csv (pandas)",
            "Збереження у Parquet (pandas)",
            "Читання flights_clean.parquet (pandas)",
            "Читання flights_clean.parquet (polars)",
            "Розмір вихідного CSV",
            "Розмір чистого Parquet",
        ],
        "Значення": [
            f"{t_load_csv} с",
            f"{t_save_parquet} с",
            f"{t_load_parquet_pd} с",
            f"{t_load_parquet_pl} с",
            f"{size_csv} МБ",
            f"{size_parquet} МБ",
        ],
    }
)

benchmarks

,Операція / Формат,Значення
0,Читання flights.csv (pandas),16.2 с
1,Збереження у Parquet (pandas),1.43 с
2,Читання flights_clean.parquet (pandas),0.55 с
3,Читання flights_clean.parquet (polars),0.23 с
4,Розмір вихідного CSV,564.96 МБ
5,Розмір чистого Parquet,30.91 МБ
